# 04주 · 단일 관로·펌프·에너지 방정식

수리학 CE272 · 주차별 공통 실습

핵심: 원리·식의 의미·손계산·조건 변경·결과 해석. 입력 단위와 지배 방정식·경계조건부터 확인

학습 주제: 퀴즈 #1·단일 관로 해석 / 수조·펌프·터빈

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/week04/CE272_W04.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE272-hydraulics/week04) · [계산 도구·가정](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/METHODS.md)

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)

실행: 처음에는 위에서 아래로 셀 실행. 공통 준비 셀 실행 후 각 S절 개별 실행 가능

진행: 예측 → 손계산·식 설정 → 실행 → 조건 하나 변경 → 원리·가정·결과 설명

Excel 중점: Type II 유량 반복·펌프 동력

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import importlib.util
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.integrate import trapezoid

# 로컬 공통 자료에서는 옆 과목 파일 사용. Colab에서는 같은 공개 모듈 불러오기
module_path = next((p for p in [Path.cwd()/"hydraulics.py", Path.cwd().parent/"hydraulics.py"] if p.exists()), None)
if module_path is None:
    module_path = Path(tempfile.gettempdir())/"lecture_ce272_hydraulics.py"
    urlretrieve("https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/hydraulics.py",module_path)
spec = importlib.util.spec_from_file_location("lecture_ce272_hydraulics",module_path)
hyd = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hyd)
print("수리학 공통 계산 도구 준비 완료. SI 단위·Darcy 계수 사용")

## S01. 수두·유량·직경의 세 문제 유형

**슬라이드 연결**: CE272_W04a_Pipe-SinglePipe.pptx p5, 6, 7, 8, 9, 10, 11, 12, 13, 14

**자료 구분**: 원본 Type I·II·III

**먼저 예측**: 주어진 손실수두와 주어진 저수지 높이차는 항상 같은가?

**문제**: Type I의 Q=0.0084 m³/s, L=100 m, D=0.075 m, 매끈한 관, 입구 K=0.5로 필요한 저수지 수심을 구하시오. Type II의 H=24 m, L=180 m, D=0.1 m, ε=0.25 mm로 자유방류 유량을 구하시오. Type III의 Q=0.5 m³/s, L=500 m, 마찰손실 2 m인 매끈한 관의 직경을 구하시오. ν=1.006×10⁻⁶ m²/s를 사용하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교

In [ ]:
nu = 1.006e-6
Q1, D1 = .0084, .075
V1 = Q1/hyd.area_pipe(D1)
H1 = hyd.pipe_loss(Q1,100,D1,nu,minor_K=.5) + V1**2/(2*hyd.G)
# 자유방류 끝단의 속도수두를 손실항과 별도로 산입
Q2 = hyd.turbulent_flow(24,[dict(L=180,D=.1,epsilon=.00025)],nu,exit_coefficient=1)
D3 = brentq(lambda D:hyd.pipe_loss(.5,500,D,nu)-2,.1,2.)
pd.Series({'Type_I_required_depth_m':H1,'Type_II_flow_m3_s':Q2,
           'Type_III_diameter_m':D3,'Type_III_loss_check_m':hyd.pipe_loss(.5,500,D3,nu)})

**검토 문제**: 자유방류 속도수두와 큰 수조로 유입될 때의 출구손실을 중복해서 넣지 않는 방법을 설명하시오. 각 해의 Reynolds 수와 에너지 잔차를 확인하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분

## S02. 펌프의 수동력과 축동력

**슬라이드 연결**: CE272_W04b_Pipe-SinglePipe.pptx p18, 19, 20, 21

**자료 구분**: 원본 펌프 예제·효율 추가 실험

**먼저 예측**: 유량·관로·높이차만으로 펌프의 축동력까지 결정할 수 있는가?

**문제**: 높이차 30 m인 수조 사이에 Q=0.3 m³/s를 양수한다. L₁=10 m, D₁=0.4 m, L₂=100 m, D₂=0.3 m이며 두 관의 조도는 0.2 mm이다. 국부손실을 무시하고 필요한 펌프수두와 수동력을 구하시오. 효율 0.8을 추가로 가정하여 축동력을 비교하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교

In [ ]:
Q, H_static, nu, eta = .3, 30., 1.006e-6, .8
hf1 = hyd.pipe_loss(Q,10,.4,nu,.0002)
hf2 = hyd.pipe_loss(Q,100,.3,nu,.0002)
H_pump = H_static+hf1+hf2
P_water = 1000*hyd.G*Q*H_pump
pd.Series({'loss_1_m':hf1,'loss_2_m':hf2,'pump_head_m':H_pump,
           'water_power_kW':P_water/1000,'shaft_power_kW_eta_0_8':P_water/eta/1000})

**검토 문제**: 추가 가정인 효율을 원본 입력과 구분하시오. 유량을 바꿀 때 마찰손실과 동력이 어떻게 달라지는지 설명하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분

## 학습 기록

두 에너지 단면을 먼저 정하고, 속도수두·국부손실·펌프수두를 구분하시오.

선택 심화: 개념 학습 후 AI의 설명·계산 검토. 단위·가정·보존식으로 확인한 근거와 수정한 부분 기록

In [ ]:
prediction = None
hand_calculation = None
changed_condition = None
principle_and_limits = None
records = {'예측':prediction,'손계산':hand_calculation,'조건 변경':changed_condition,'원리·한계':principle_and_limits}
missing = [k for k,v in records.items() if v is None or str(v).strip() == '']
print('작성 필요:',missing) if missing else print('기록 항목 작성 완료. 계산·논리는 별도 검토 필요')